# 05 - Structure fusion

Add **AlphaFold/ESMFold confidence** features (pTM, ipTM, PAE at the interface,
pLDDT) on top of the sequence representation. We load the bundled precomputed
confidence summaries (no GPU folding), train the **fusion** model if torch is
available, and measure the **lift on cliff records** versus the baseline.

In [ ]:
import numpy as np
from tcr_cliff.data import load_toy, split_pairs, load_toy_structure_summary
from tcr_cliff.config import Config, DataConfig, EmbeddingConfig, CliffConfig, ModelConfig, StructureConfig
from tcr_cliff.models import train_model, predict_scores
from tcr_cliff.eval import cliff_aware_report, compare_models
from tcr_cliff.cliffs import find_neighbor_pairs

df = load_toy()
parts = split_pairs(df, DataConfig(group_split_on='peptide', split_column='__none__'), seed=0)
train_df, test_df = parts['train'], parts['test']

## Load the structure-confidence features

`load_structure_features` returns a `[N, 12]` matrix keyed by `pair_id`. With
`use_stub=True` (default) it reads the bundled summaries; records without an entry
get a zero vector with `available_flag=0`.

In [ ]:
from tcr_cliff.structure import load_structure_features, STRUCTURE_FEATURE_NAMES

struct_cfg = StructureConfig(enabled=True, use_stub=True)
S, snames = load_structure_features(test_df, struct_cfg)
print('structure features:', S.shape)
print('feature names (12):', STRUCTURE_FEATURE_NAMES)
assert S.shape[1] == 12 == len(STRUCTURE_FEATURE_NAMES)

### Confidence separates binders from non-binders

Binders fold into a confident interface (high ipTM, low interface PAE); decoys do
not. We show this directly from the bundled summary dict.

In [ ]:
summary = load_toy_structure_summary()
iptm_by_label = {0: [], 1: []}
for _, row in df.iterrows():
    rec = summary.get(row['pair_id'])
    if rec is not None:
        iptm_by_label[int(row['binder'])].append(rec['iptm'])
for label in (0, 1):
    vals = iptm_by_label[label]
    print(f'binder={label}: mean ipTM = {np.mean(vals):.3f}  (n={len(vals)})')

## Baseline (sequence only) for reference

In [ ]:
base_cfg = Config(
    seed=0,
    embedding=EmbeddingConfig(backend='fallback', fallback_dim=64, cache_dir=None),
    cliff=CliffConfig(max_edits=1, vary='both'),
    structure=struct_cfg,
    model=ModelConfig(kind='baseline_lgbm'),
)
base_cfg.model.baseline.n_estimators = 50

test_pairs = find_neighbor_pairs(test_df, base_cfg.cliff)
baseline = train_model(base_cfg, train_df)
y_base = predict_scores(baseline, test_df)
rep_base = cliff_aware_report(test_df, y_base, pairs=test_pairs)
print('baseline cliff-record AUROC:', round(rep_base['cliff_records']['auroc'], 3))

## Train the fusion model **if torch is available**

The fusion head concatenates the cliff-aware sequence representation with the 12
structure-confidence features before the binding classifier.

In [ ]:
reports = {'baseline (seq)': rep_base}

try:
    import torch  # noqa: F401
    torch_available = True
except ImportError:
    torch_available = False

if torch_available:
    fus_cfg = base_cfg.model_copy(deep=True)
    fus_cfg.model = ModelConfig(kind='fusion')
    fus_cfg.model.cliff_aware.proj_dim = 32
    fus_cfg.model.cliff_aware.hidden_dim = 64
    fus_cfg.model.cliff_aware.contrastive_epochs = 3
    fus_cfg.model.cliff_aware.head_epochs = 5
    fus_cfg.structure = struct_cfg
    fusion = train_model(fus_cfg, train_df)
    y_fus = predict_scores(fusion, test_df)
    reports['fusion (seq+struct)'] = cliff_aware_report(test_df, y_fus, pairs=test_pairs)
    print('fusion cliff-record AUROC:',
          round(reports['fusion (seq+struct)']['cliff_records']['auroc'], 3))
else:
    print('torch not available - skipping fusion model.')
    print('Install with:  pip install "tcr-cliff[torch]"')
    print('Fusion adds the 12 structure-confidence features above to the sequence head.')

## Measure the lift on cliff records

In [ ]:
table = compare_models(reports)
table

In [ ]:
if 'fusion (seq+struct)' in reports:
    lift = (reports['fusion (seq+struct)']['cliff_records']['auroc']
            - reports['baseline (seq)']['cliff_records']['auroc'])
    print(f'structure-fusion lift on cliff-record AUROC: {lift:+.3f}')
else:
    print('Run with torch installed to measure the structure-fusion lift.')

### Next
Continue to **06_interpret** to see *which residues* drive the prediction.